# Libraries

In [ ]:
from graph_creation import create_mesh_dataset, save_database, create_dataset_folders

In [ ]:
create_dataset_folders(dataset_folder='datasets')

# Create and save pytorch geometric dataset

In [ ]:
# dataset_folder, dataset_name, dataset_dir, with_multiscale, start_sim_id, n_sim
simulation_ids = [
    # ['./raw_datasets_mesh', 'mesh_dataset2','datasets/train', False, range(1, 1+80)],
    # ['./raw_datasets_mesh', 'mesh_dataset2','datasets/test', False, range(81, 81+20)],
    # ['./raw_datasets_mesh', 'multiscale_mesh_dataset2','datasets/train', True, range(1, 1+80)],
    # ['./raw_datasets_mesh', 'multiscale_mesh_dataset2','datasets/test', True, range(81, 81+1)],
    # ['./raw_datasets_dk15', 'dijkring_15_big','datasets/test', True, range(101, 101+11)],
    # ['./raw_datasets_dk15', 'dijkring_15_big_fine','datasets/test', False, range(101, 101+1)],
    # ['./raw_datasets_dyce', 'dyce_lisfloodfp_cpu_highflow1', 'datasets/train', True, [0, 1, 3, 4, 7 , 9, 10, 11, 12]],
    # ['./raw_datasets_dyce', 'dyce_lisfloodfp_cpu_highflow2', 'datasets/train', True, [13, 15, 16, 18, 19, 21, 22, 23, 25]],
    # ['./raw_datasets_dyce', 'dyce_lisfloodfp_cpu_highflow3', 'datasets/train', True, [26, 28, 29, 30, 32, 33, 34, 36, 37]],
    # ['./raw_datasets_dyce', 'dyce_lisfloodfp_cpu_highflow4', 'datasets/train', True, [38, 39, 40, 42, 43, 44, 45, 46, 47]],
    ['./raw_datasets_dyce', 'dyce_lisfloodfp_cpu_highflow1test', 'datasets/test', True, [2, 5, 6, 8, 14, 17, 20, 24]],
    ['./raw_datasets_dyce', 'dyce_lisfloodfp_cpu_highflow2test', 'datasets/test', True, [27, 31, 35, 41, 48, 49, 50, 51]],
]

In [ ]:
for dataset_folder, dataset_name, dataset_dir, with_multiscale, sim_ids in simulation_ids:
   # sim_ids = range(start_sim_id, start_sim_id+n_sim)
   mesh_dataset = create_mesh_dataset(dataset_folder, sim_ids=sim_ids,
                                      with_multiscale=with_multiscale, number_of_multiscales=5,
                                      netcdf_file_template='hydrograph_{:04}_highflow.nc.zst', DEM_file_template='dyce_lisfloodfp.xyz',
                        hydrograph_file_template='hydrograph_{:04}_highflow.txt', polygon_file_template='dyce_polygon.pol'
                                      )
   
   if dataset_name[:8] == 'dijkring':
      train_dataset = [mesh_dataset[0]]
      test_dataset = mesh_dataset[1:]
      save_database(train_dataset, name=dataset_name, out_path='datasets/train')
      save_database(test_dataset, name=dataset_name, out_path='datasets/test')
   else:
      save_database(mesh_dataset, name=dataset_name, out_path=dataset_dir)

##############


# Create and save mesh only dataset
Procedure for creating a dataset containing a prepared mesh, but no depth, velocity or temporal boundary condition data.

In [ ]:
import importlib
import pickle

import graph_creation
importlib.reload(graph_creation)
from graph_creation import export_zarr_dataset, MultiscaleMesh

VECTOR_BC = True

raw_meshes = pickle.load(open("raw_datasets_dyce/dycecut_mesh.pkl","rb"))[4:6]

In [ ]:
if VECTOR_BC:
    import numpy as np
    import xarray as xr
    num_nodes_BC = (xr.open_zarr('mesh_2meshes_cut_withBC.zarr').mesh2d_edge_type == 2).sum().values # Number of boundary nodes
    type_BC = np.full(num_nodes_BC, 1) # Set all boundary nodes to water level boundaries
    type_BC[4] = 2 # Set source node to water discharge boundary instead 
else:
    type_BC = 2

In [ ]:
mesh_dataset = graph_creation.create_mesh_dataset(
    "raw_datasets_dyce", sim_ids=[None],
    with_multiscale=True, number_of_multiscales=2,
    netcdf_file_template='../../mesh_2meshes_cut_withBC.zarr', DEM_file_template='dyce_lisfloodfp.xyz',
    hydrograph_file_template=None, polygon_file_template='dycecut_polygon.pol',
    multiscale_mesh_file=None, raw_meshes=raw_meshes, type_BC=type_BC, template_only=True
)[0]
graph_creation.save_database(mesh_dataset, name=f"dataset_dyce_2mesh_cut_withBC", out_path="raw_datasets_dyce", zarr_output=False)

Visualise mesh, edge indices and added ghost faces:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib qt

mesh = mesh_dataset.mesh.meshes[0]

fig, ax = plt.subplots()
ghost_cells_mask = np.zeros(mesh.face_x.shape[0])
ghost_cells_mask[mesh.ghost_cells_ids] = 1
graph_creation.plot_faces(mesh, ax, face_value=ghost_cells_mask)
for i, bc_edge in enumerate(mesh.edge_index_BC):
    colour = "red" if mesh_dataset.type_BC[i] == 1 else "green"
    ax.plot(mesh.node_x[bc_edge], mesh.node_y[bc_edge], color=colour)
    ax.annotate(f"{i},{bc_edge}", xy=(np.mean(mesh.node_x[bc_edge]), np.mean(mesh.node_y[bc_edge])), color=colour )
plt.show()

# ZARR Dataset Creation
OLD: for creation of .zarr files from .nc files

In [ ]:
# for dataset_folder, dataset_name, dataset_dir, with_multiscale, sim_ids in simulation_ids:
#     for sim_id in sim_ids:
#         print(f"hydrograph_{sim_id:04}_highflow")
#         mesh_dataset = create_mesh_dataset(
#             dataset_folder, sim_ids=[sim_id],
#             with_multiscale=with_multiscale, number_of_multiscales=5,
#             netcdf_file_template='hydrograph_{:04}_highflow.zarr', DEM_file_template='dyce_lisfloodfp.xyz',
#             hydrograph_file_template='hydrograph_{:04}_highflow.txt', polygon_file_template='dyce_polygon.pol',
#             multiscale_mesh_file="./raw_datasets_dyce/dyce_mesh.pkl"
#         )
#         save_database(mesh_dataset[0], name=f"hydrograph_{sim_id:04}_highflow", out_path="datasets/train/dyce/", zarr_output=True)

##############
from graph_creation import add_ghost_cells_attributes, export_zarr_dataset, pool_multiscale_attributes
import pickle
import torch

for dataset_folder, dataset_name, dataset_dir, with_multiscale, sim_ids in simulation_ids:
    data = pickle.load(open("./raw_datasets_dyce/dataset_dyce.pkl", "rb"))
    for sim_id in sim_ids:
        print(f"hydrograph_{sim_id:04}_highflow")
        simulation_dataset = xr.open_dataset(f"./raw_datasets_dyce/Simulations_6meshes/hydrograph_{sim_id:04}_highflow.zarr")

        WD = simulation_dataset['mesh2d_waterdepth'].data.T
        VX = simulation_dataset['mesh2d_ucx'].data.T
        VY = simulation_dataset['mesh2d_ucy'].data.T
        
        BC = np.loadtxt(f"./raw_datasets_dyce/Hydrograph/hydrograph_{sim_id:04}_highflow.txt")
        BC[:,0] /= 60 # convert to minutes
        BC = torch.FloatTensor(BC).unsqueeze(0).repeat(len(data.node_BC), 1, 1) # This repeats the same BC
        
        DEM = data.mesh.DEM
        DEM, WD, VX, VY = add_ghost_cells_attributes(data.mesh.meshes[0], DEM, WD, VX, VY) # Was originally immediately before mesh.stack_meshes(meshes), but placing it here is logically equivalent
        DEM, WD, VX, VY = pool_multiscale_attributes(data.mesh, DEM, WD, VX, VY, reduce='mean')

        export_zarr_dataset({"WD":WD, "VX":VX, "VY":VY, "BC":BC}, path=f"datasets/train/dyce/hydrograph_{sim_id:04}_highflow.zarr")
        # save_database(mesh_dataset[0], name=f"hydrograph_{sim_id:04}_highflow", out_path="datasets/train/dyce/", zarr_output=True)